In [ ]:
import pandas as pd

# Note: The Adidas Excel file often has a few empty rows at the top.
df = pd.read_csv('/content/train.csv')

# Display the first 5 rows to see what we are working with
df.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30


In [ ]:
# Check how many null values exist in each column
print("Null values per column:")
print(df.isnull().sum())

# Check basic info to see data types (Objects vs Floats/Ints)
df.info()

Null values per column:
ID                             0
Delivery_person_ID             0
Delivery_person_Age            0
Delivery_person_Ratings        0
Restaurant_latitude            0
Restaurant_longitude           0
Delivery_location_latitude     0
Delivery_location_longitude    0
Order_Date                     0
Time_Orderd                    0
Time_Order_picked              0
Weatherconditions              0
Road_traffic_density           0
Vehicle_condition              0
Type_of_order                  0
Type_of_vehicle                0
multiple_deliveries            0
Festival                       0
City                           0
Time_taken(min)                0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45593 entries, 0 to 45592
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           45593 non-null  object 
 1   Delivery_person

In [ ]:
import pandas as pd
import numpy as np

input_file = '/content/train.csv'
output_file = "clean_food_delivery.csv"

df = pd.read_csv(input_file)

print("Original shape:", df.shape)

df = df.replace(
    ["NaN", "nan", "NA", "N/A", ""],
    np.nan
)


df["Delivery_person_Age"] = pd.to_numeric(
    df["Delivery_person_Age"],
    errors="coerce"
)

df["Delivery_person_Ratings"] = pd.to_numeric(
    df["Delivery_person_Ratings"],
    errors="coerce"
)

df["multiple_deliveries"] = pd.to_numeric(
    df["multiple_deliveries"],
    errors="coerce"
)

df["Vehicle_condition"] = pd.to_numeric(
    df["Vehicle_condition"],
    errors="coerce"
)


df["Time_taken(min)"] = (
    df["Time_taken(min)"]
    .astype(str)
    .str.extract(r"(\d+\.?\d*)")[0]
)

df["Time_taken(min)"] = pd.to_numeric(
    df["Time_taken(min)"],
    errors="coerce"
)


# Check the result
print("\nTime_taken(min) after cleaning:")
print(df["Time_taken(min)"].head())


df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

order_time = pd.to_timedelta(
    df["Time_Orderd"],
    errors="coerce"
)

pickup_time = pd.to_timedelta(
    df["Time_Order_picked"],
    errors="coerce"
)


df["Order_Timestamp"] = (
    df["Order_Date"] + order_time
)

df["Pickup_Timestamp"] = (
    df["Order_Date"] + pickup_time
)


midnight_mask = (
    df["Pickup_Timestamp"] <
    df["Order_Timestamp"]
)

df.loc[
    midnight_mask,
    "Pickup_Timestamp"
] += pd.Timedelta(days=1)

print("\nMissing values after cleaning:")

print(
    df[
        [
            "Order_Timestamp",
            "Pickup_Timestamp",
            "Time_taken(min)"
        ]
    ].isna().sum()
)


required_columns = [
    "Order_Timestamp",
    "Pickup_Timestamp",
    "Time_taken(min)"
]

before = len(df)

df = df.dropna(
    subset=required_columns
)

print(
    "\nRows removed because of invalid/missing data:",
    before - len(df)
)

print(
    "Rows remaining:",
    len(df)
)


df["Prep Time"] = (
    df["Pickup_Timestamp"] -
    df["Order_Timestamp"]
).dt.total_seconds() / 60


df["Total Delivery Time"] = (
    df["Time_taken(min)"]
)


df["Transit Time"] = (
    df["Total Delivery Time"] -
    df["Prep Time"]
)


before = len(df)

df = df[
    (df["Prep Time"] >= 0) &
    (df["Transit Time"] >= 0) &
    (df["Total Delivery Time"] > 0)
]

print(
    "\nInvalid time rows removed:",
    before - len(df)
)


def remove_outliers_iqr(data, column):

    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    print(f"\n{column} outlier limits:")
    print("Q1:", round(Q1, 2))
    print("Q3:", round(Q3, 2))
    print("IQR:", round(IQR, 2))
    print("Lower:", round(lower_bound, 2))
    print("Upper:", round(upper_bound, 2))

    return data[
        (data[column] >= lower_bound) &
        (data[column] <= upper_bound)
    ]


print("\nRemoving outliers...")

df = remove_outliers_iqr(
    df,
    "Total Delivery Time"
)


hour = df["Order_Timestamp"].dt.hour


def classify_time(hour):

    if 6 <= hour < 11:
        return "Morning"

    elif 11 <= hour < 14:
        return "Lunch Peak"

    elif 14 <= hour < 17:
        return "Afternoon"

    elif 17 <= hour < 21:
        return "Dinner Rush"

    elif 21 <= hour < 24:
        return "Late Night"

    else:
        return "Overnight"


df["Time of Day"] = hour.apply(
    classify_time
)


DELAY_THRESHOLD = 30

df["Delayed Order"] = np.where(
    df["Total Delivery Time"] > DELAY_THRESHOLD,
    1,
    0
)

df["Delay Status"] = np.where(
    df["Delayed Order"] == 1,
    "Delayed",
    "On Time"
)


df["Prep Time"] = df["Prep Time"].round(2)
df["Transit Time"] = df["Transit Time"].round(2)
df["Total Delivery Time"] = (
    df["Total Delivery Time"].round(2)
)


df = df.drop(
    columns=[
        "Order_Timestamp",
        "Pickup_Timestamp"
    ]
)


print("\n==========================================")
print("FINAL DATASET")
print("==========================================")

print("Final shape:", df.shape)

print("\nNew columns:")
print(
    [
        "Prep Time",
        "Transit Time",
        "Total Delivery Time",
        "Time of Day",
        "Delayed Order",
        "Delay Status"
    ]
)


print("\nSample of processed data:")

print(
    df[
        [
            "ID",
            "Time_Orderd",
            "Time_Order_picked",
            "Time_taken(min)",
            "Prep Time",
            "Transit Time",
            "Total Delivery Time",
            "Time of Day",
            "Delayed Order"
        ]
    ].head(10)
)



df.to_csv(
    output_file,
    index=False
)


print(
    f"Saved to: {output_file}"
)

Original shape: (45593, 20)

Time_taken(min) after cleaning:
0    24
1    33
2    26
3    21
4    30
Name: Time_taken(min), dtype: int64

Missing values after cleaning:
Order_Timestamp     1731
Pickup_Timestamp       0
Time_taken(min)        0
dtype: int64

Rows removed because of invalid/missing data: 1731
Rows remaining: 43862

Invalid time rows removed: 1243

Removing outliers...

Total Delivery Time outlier limits:
Q1: 19.0
Q3: 33.0
IQR: 14.0
Lower: -2.0
Upper: 54.0

FINAL DATASET
Final shape: (42619, 26)

New columns:
['Prep Time', 'Transit Time', 'Total Delivery Time', 'Time of Day', 'Delayed Order', 'Delay Status']

Sample of processed data:
        ID Time_Orderd Time_Order_picked  Time_taken(min)  Prep Time  \
0  0x4607     11:30:00          11:45:00               24       15.0   
1  0xb379     19:45:00          19:50:00               33        5.0   
2  0x5d6d     08:30:00          08:45:00               26       15.0   
3  0x7a6a     18:00:00          18:10:00               